# Aula 04 — Banco de dados e SQL

Sair da lista em memória e guardar as vendas em um **banco relacional**: esquema com chaves
estrangeiras, inserção parametrizada, `JOIN`, `GROUP BY`, subquery, window function e transação.

As operações testáveis vivem em `src/opencode_harness/banco.py`; os testes, em `tests/test_banco.py`.
Aqui usamos `:memory:` — o banco nasce e morre com o notebook, sem arquivo para versionar.

## 1. Conexão

Duas configurações fazem diferença desde o início: `row_factory` (acessar coluna pelo nome) e `PRAGMA foreign_keys = ON` (o SQLite **não** valida FK por padrão).

In [ ]:
from __future__ import annotations

import sqlite3

from opencode_harness import (
    conectar,
    criar_esquema,
    inserir_cliente,
    inserir_produto,
    registrar_venda,
    ticket_medio,
    top_clientes,
)
from opencode_harness.banco import receita_por_categoria

conn = conectar()
print("SQLite", sqlite3.sqlite_version)
print("foreign_keys ligado:", conn.execute("PRAGMA foreign_keys").fetchone()[0] == 1)

## 2. Esquema

Três tabelas: `clientes`, `produtos` (com `UNIQUE` e `CHECK`) e `vendas`, que liga as duas por chave estrangeira. Os índices cobrem as buscas mais comuns.

In [ ]:
criar_esquema(conn)

tabelas = conn.execute(
    "SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name"
).fetchall()
indices = conn.execute(
    "SELECT name FROM sqlite_master WHERE type = 'index' AND name LIKE 'idx_%' ORDER BY name"
).fetchall()

print("tabelas:", [linha["name"] for linha in tabelas])
print("índices:", [linha["name"] for linha in indices])
print()
print(conn.execute("SELECT sql FROM sqlite_master WHERE name = 'vendas'").fetchone()["sql"])

## 3. INSERT parametrizado

Os valores vão como `?` e entram na tupla: o driver escapa, então aspas e `;` não viram comando. `with conn:` faz **commit** no fim e **rollback** se qualquer inserção falhar.

In [ ]:
with conn:
    inserir_cliente(conn, 1, "Ana", "SP")
    inserir_cliente(conn, 2, "Bia", "RJ")
    inserir_produto(conn, 1, "notebook", "hardware", 3500.0)
    inserir_produto(conn, 2, "mouse", "periferico", 80.0)
    inserir_produto(conn, 3, "teclado", "periferico", 150.0)
    inserir_produto(conn, 4, "monitor", "hardware", 1200.0)

    registrar_venda(conn, 1, 1, 2, "2026-08-01")
    registrar_venda(conn, 1, 2, 10, "2026-08-02")
    registrar_venda(conn, 2, 4, 1, "2026-09-05")
    registrar_venda(conn, 2, 3, 4, "2026-09-06")
    registrar_venda(conn, 1, 3, 2, "2026-10-10")

contagens = {
    "clientes": conn.execute("SELECT COUNT(*) FROM clientes").fetchone()[0],
    "produtos": conn.execute("SELECT COUNT(*) FROM produtos").fetchone()[0],
    "vendas": conn.execute("SELECT COUNT(*) FROM vendas").fetchone()[0],
}
for tabela, total in contagens.items():
    print(f"{tabela:9s} {total} linha(s)")

## 4. SELECT, WHERE e ORDER BY

Filtrar antes de agregar: só os produtos de hardware, do mais caro para o mais barato.

In [ ]:
sql = """
    SELECT nome, categoria, preco
    FROM produtos
    WHERE categoria = ?
    ORDER BY preco DESC
"""

for linha in conn.execute(sql, ("hardware",)):
    print(f"{linha['nome']:10s} R$ {linha['preco']:8.2f}")

## 5. JOIN + GROUP BY

A receita não está gravada em lugar nenhum: ela é `quantidade * preco`, calculada na consulta. O `JOIN` traz o preço do produto e o `GROUP BY` soma por categoria.

In [ ]:
for categoria, receita in receita_por_categoria(conn):
    print(f"{categoria:12s} R$ {receita:9.2f}")

print(f"ticket médio: R$ {ticket_medio(conn):.2f}")

## 6. HAVING e subquery

`WHERE` filtra **linhas**, `HAVING` filtra **grupos**. E a subquery permite comparar cada venda com a média geral.

In [ ]:
sql = """
    SELECT p.categoria, ROUND(SUM(v.quantidade * p.preco), 2) AS receita
    FROM vendas AS v
    JOIN produtos AS p ON p.id = v.produto_id
    GROUP BY p.categoria
    HAVING receita > ?
    ORDER BY receita DESC
"""
print("categorias acima de R$ 2.000:", [tuple(linha) for linha in conn.execute(sql, (2000,))])

sql = """
    SELECT v.id, p.nome, ROUND(v.quantidade * p.preco, 2) AS total
    FROM vendas AS v
    JOIN produtos AS p ON p.id = v.produto_id
    WHERE v.quantidade * p.preco > (
        SELECT AVG(v2.quantidade * p2.preco)
        FROM vendas AS v2
        JOIN produtos AS p2 ON p2.id = v2.produto_id
    )
    ORDER BY total DESC
"""
for linha in conn.execute(sql):
    print(f"venda {linha['id']}: {linha['nome']:9s} R$ {linha['total']:8.2f}")

## 7. Window function

`RANK() OVER (ORDER BY total DESC)` numera os clientes **sem colapsar as linhas**, como o `GROUP BY` faria. É o mesmo mecanismo por trás de rankings em dashboards.

In [ ]:
for nome, total, posicao in top_clientes(conn, n=2):
    print(f"{posicao}º {nome:5s} R$ {total:8.2f}")

## 8. Transação e rollback

O banco barra o que viola o contrato: `CHECK` para quantidade, `FOREIGN KEY` para cliente inexistente. E nada fica gravado pela metade.

In [ ]:
try:
    with conn:
        registrar_venda(conn, 1, 1, 0, "2026-11-01")  # quantidade > 0 (CHECK)
except sqlite3.IntegrityError as erro:
    print("bloqueado pelo CHECK:", erro)

try:
    with conn:
        registrar_venda(conn, 99, 1, 1, "2026-11-02")  # cliente 99 não existe (FK)
except sqlite3.IntegrityError as erro:
    print("bloqueado pela FK:", erro)

print("vendas depois dos rollbacks:", conn.execute("SELECT COUNT(*) FROM vendas").fetchone()[0])

## 9. Índice e plano de execução

`EXPLAIN QUERY PLAN` mostra o que o motor vai fazer. `SEARCH ... USING INDEX` é o sinal de que o índice foi usado; `SCAN` significa varrer a tabela inteira.

In [ ]:
com_indice = "EXPLAIN QUERY PLAN SELECT * FROM vendas WHERE cliente_id = 1"
sem_indice = "EXPLAIN QUERY PLAN SELECT * FROM vendas WHERE quantidade > 1"

for titulo, sql in (("por cliente", com_indice), ("por quantidade", sem_indice)):
    print(titulo + ":")
    for linha in conn.execute(sql):
        print(" ", linha["detail"])
    print()

## Conclusões

O que treinamos aqui:

- **esquema antes de dado**: `PRIMARY KEY`, `FOREIGN KEY`, `UNIQUE` e `CHECK` são regras, não decoração;
- **`PRAGMA foreign_keys = ON`**: sem isso o SQLite aceita órfão silenciosamente;
- **consulta parametrizada** (`?` + tupla): única forma segura de interpolar valor em SQL;
- **`JOIN` + `GROUP BY`**: receita calculada na hora, sem coluna redundante;
- **`WHERE` filtra linha, `HAVING` filtra grupo**;
- **subquery** para comparar cada linha com um agregado;
- **window function** para ranking sem perder o grão da linha;
- **transação** (`with conn:`): ou tudo entra, ou nada entra;
- **índice** existe para a consulta que você realmente faz — o plano de execução mostra se ele foi usado.

> **SQLite x banco de produção:** aqui o banco é um arquivo único e sem usuário/senha. Em Postgres
> ou MySQL os tipos são mais rígidos (`NUMERIC` para dinheiro, `DATE` de verdade) e o `EXPLAIN` é
> mais detalhado. O SQL que você escreveu aqui continua válido em quase tudo.

**Próximos passos**

1. Módulo 05 — Análise exploratória: carregar mais linhas no banco e responder perguntas com `pandas` (`pd.read_sql`).
2. Guardar o banco em arquivo (`data/processed/vendas.db`, fora do git) em vez de `:memory:`.
3. Adicionar `VIEW` para a receita por categoria e comparar com a consulta direta.

**Checklist de entrega**

- [x] Notebook executa do início ao fim sem erro
- [x] `uv run pytest -q` cobre esquema, constraints e agregações
- [x] `uv run ruff check .` limpo
- [ ] PR aberto e revisão automática lida
- [ ] Merge feito e módulo marcado em `docs/DIO-ROADMAP.md`